# 01. Inspect model parts and supervision

This model has an InternViT vision encoder, a two-linear-layer `mlp1` projector and a Qwen2
language decoder. These are the three independently selectable LoRA targets.
Start with rank 4, q/v adapters in the decoder's last eight blocks. Freeze the image encoder,
projector, embeddings and output head. Gradients should reach only adapter parameters.

The final cells load the model **only when you manually run them**. Close other kernels first:
the GPU has 8 GiB and Windows needs paging-file headroom and free disk space.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "models.json").exists():
    raise FileNotFoundError("Start the kernel in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
print("Interpreter:", sys.executable)
print("Dataset:", H.PAGE_DIR)
print("Local base model:", H.MODEL_DIR)


In [ ]:
from importlib.metadata import version, PackageNotFoundError
for package in ("torch", "transformers", "peft", "accelerate", "safetensors"):
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "MISSING")
config = json.loads((H.MODEL_DIR / "config.json").read_text(encoding="utf-8"))
display({"decoder": config["llm_config"]["architectures"],
         "decoder_blocks": config["llm_config"]["num_hidden_layers"],
         "vision_blocks": config["vision_config"]["num_hidden_layers"],
         "projector": "LayerNorm -> Linear (1) -> GELU -> Linear (3)",
         "image_size": config["force_image_size"]})


In [ ]:
model, tokenizer = H.load_base()
selected = H.experiment("inspect_only", decoder="qv", last_layers=8, rank=4)
display(H.attach_lora(model, selected))
names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
print("Adapter tensors:", len(names))
display(names[:20])
assert all("lora_" in name for name in names)


Inspect assistant-only masking and image-token alignment before any optimization.
The training loss computes the vocabulary head only for the answer suffix, reducing memory.
It uses the downloaded model's conversation template, image token expansion and causal shift.
Only the assistant answer and closing separator are supervised. Images/answers are not truncated.


In [ ]:
pages = H.read_pages()  # Requires manually reviewed labels from 00.
sample = H.partition(pages, "smoke")[0]
batch = H.example(model, tokenizer, sample, selected)
print("Image tiles (including optional thumbnail):", len(batch["pixel_values"]))
print("Total tokens:", batch["input_ids"].shape[1])
print("Masked prompt tokens:", batch["answer_start"])
print("Supervised answer:", tokenizer.decode(batch["labels"][0, batch["answer_start"]:]))
assert (batch["labels"][0, :batch["answer_start"]] == -100).all()
assert (batch["labels"][0, batch["answer_start"]:] >= 0).all()
del batch, model, tokenizer
H.cleanup()
